In [1]:
import pickle
import numpy as np

BASE = "/kaggle/input/datasets/ayushaabbas/patchcore-outputs/kaggle/working"
with open(f"{BASE}/patchcore_full_results.pkl", "rb") as f:
    results = pickle.load(f)
memory_banks = {}
for cat in results.keys():
    memory_banks[cat] = np.load(f"{BASE}/membank_{cat}.npy")
print("✓ Categories:", list(results.keys()))
print("✓ Memory banks:", {k: v.shape for k, v in memory_banks.items()})

✓ Categories: ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']
✓ Memory banks: {'bottle': (1638, 1536), 'cable': (1756, 1536), 'capsule': (1716, 1536), 'carpet': (2195, 1536), 'grid': (2069, 1536), 'hazelnut': (3065, 1536), 'leather': (1920, 1536), 'metal_nut': (1724, 1536), 'pill': (2093, 1536), 'screw': (2508, 1536), 'tile': (1803, 1536), 'toothbrush': (470, 1536), 'transistor': (1669, 1536), 'wood': (1936, 1536), 'zipper': (1881, 1536)}


In [2]:
from PIL import Image
import torch
import torch.nn as nn
import timm
import numpy as np
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
backbone = timm.create_model('wide_resnet50_2', pretrained=True, features_only=True)
backbone = backbone.to(device).eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
print("✓ Backbone ready")

model.safetensors:   0%|          | 0.00/276M [00:00<?, ?B/s]

✓ Backbone ready


In [3]:
def extract_features(image_path):
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = backbone(x)
    f2 = torch.nn.functional.interpolate(feats[2], size=(28,28), mode='bilinear', align_corners=False)
    f3 = torch.nn.functional.interpolate(feats[3], size=(28,28), mode='bilinear', align_corners=False)
    combined = torch.cat([f2, f3], dim=1)
    b, c, h, w = combined.shape
    return combined.permute(0,2,3,1).reshape(-1, c).cpu().numpy()

def build_nn(memory_bank, k=9):
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k, metric='euclidean', algorithm='ball_tree', n_jobs=-1)
    nn.fit(memory_bank)
    return nn

def score_image(image_path, nn_index):
    patches = extract_features(str(image_path))
    dists, _ = nn_index.kneighbors(patches)
    return float(np.max(dists[:, 0])), patches

def score_all(image_paths, nn_index):
    return np.array([score_image(p, nn_index)[0] for p in image_paths])

def compute_aucc(auroc_curve, baseline_auroc):
    gains = [max(0.0, a - baseline_auroc) for a in auroc_curve[1:]]
    max_possible = len(gains) * (1.0 - baseline_auroc)
    if max_possible <= 0:
        return 0.0
    return float(sum(gains) / max_possible)

print("✓ Functions ready")

✓ Functions ready


In [4]:
import pickle, numpy as np, time
from copy import deepcopy
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, roc_curve

# ── CONFIG ──────────────────────────────────────────
CATEGORY       = 'toothbrush'
BANK_CAP_LIST  = [2500, 5000, 10000]
N_ROUNDS       = 30
SEED           = 42
SAVE_DIR       = "/kaggle/working"

def run_cap_ablation(max_bank, seed=SEED):
    rng         = np.random.default_rng(seed)
    cat_data    = results[CATEGORY]
    image_paths = cat_data['image_paths']
    labels      = np.array(cat_data['labels'])
    subtypes    = cat_data['subtypes']

    bank        = deepcopy(memory_banks[CATEGORY])
    nn_idx      = build_nn(bank)
    scores      = score_all(image_paths, nn_idx)
    baseline    = roc_auc_score(labels, scores)
    fpr0, tpr0, _ = roc_curve(labels, scores)
    print(f"    Baseline AUROC: {baseline:.4f}  (max_bank={max_bank})")

    auroc_curve = [baseline]
    bank_curve  = [len(bank)]
    corrected   = set()
    total_fp = total_fn = 0

    for r in range(N_ROUNDS):
        scores  = score_all(image_paths, nn_idx)
        tau     = np.percentile(scores[labels == 0], 90)
        preds   = (scores >= tau).astype(int)

        fp_pool = [i for i in range(len(labels))
                   if labels[i] == 0 and preds[i] == 1 and i not in corrected]
        fn_pool = [i for i in range(len(labels))
                   if labels[i] == 1 and preds[i] == 0 and i not in corrected]
        pool    = fp_pool + fn_pool

        if not pool:
            print(f"    Round {r+1:3d}: no errors — stopping early")
            while len(auroc_curve) < N_ROUNDS + 1:
                auroc_curve.append(auroc_curve[-1])
                bank_curve.append(bank_curve[-1])
            break

        idx      = int(rng.integers(len(pool)))
        true_lbl = labels[pool[idx]]
        img_path = image_paths[pool[idx]]
        sub = subtypes[pool[idx]] if subtypes is not None and len(subtypes) > 0 else 'unknown'

        if true_lbl == 0:
            new_patches = extract_features(str(img_path))
            bank = np.concatenate([bank, new_patches], axis=0)
            if len(bank) > max_bank:   # ← ablation parameter
                idxs = np.random.choice(len(bank), max_bank, replace=False)
                bank = bank[idxs]
            total_fp += 1; ctype = 'FP'
        else:
            _, patches = score_image(img_path, nn_idx)
            dists, _   = nn_idx.kneighbors(patches, n_neighbors=1)
            anchor     = patches[np.argmax(dists.squeeze())].reshape(1,-1)
            k_act      = min(5, len(bank) - 10)
            _, rm_idx  = nn_idx.kneighbors(anchor, n_neighbors=k_act)
            mask       = np.ones(len(bank), dtype=bool)
            mask[rm_idx.flatten()] = False
            bank       = bank[mask]
            total_fn += 1; ctype = 'FN'

        nn_idx = build_nn(bank)
        corrected.add(pool[idx])
        new_scores = score_all(image_paths, nn_idx)
        auroc      = roc_auc_score(labels, new_scores)
        auroc_curve.append(auroc)
        bank_curve.append(len(bank))
        print(f"    Round {r+1:3d} | cap={max_bank} | {ctype} ({sub:20s}) | "
              f"AUROC: {auroc:.4f} | bank: {len(bank)}")

    fpr1, tpr1, _ = roc_curve(labels, new_scores if 'new_scores' in dir() else scores)
    aucc  = compute_aucc(auroc_curve, baseline)
    final = auroc_curve[-1]
    print(f"\n    cap={max_bank}: AUCC={aucc:.4f}  Final={final:.4f}")

    return {
        'auroc_curve': auroc_curve, 'bank_size_curve': bank_curve,
        'baseline_auroc': baseline, 'final_auroc': final,
        'improvement': final - baseline, 'aucc': aucc,
        'total_fp_corrected': total_fp, 'total_fn_corrected': total_fn,
        'roc_fpr_baseline': fpr0.tolist(), 'roc_tpr_baseline': tpr0.tolist(),
        'roc_fpr_final': fpr1.tolist(), 'roc_tpr_final': tpr1.tolist(),
        'max_bank': max_bank,
    }

# ── RUN ─────────────────────────────────────────────
cap_results = {}
for cap in BANK_CAP_LIST:
    print(f"\n{'='*55}\n  max_bank = {cap}\n{'='*55}")
    cap_results[cap] = run_cap_ablation(cap)

with open(f"{SAVE_DIR}/hitl_ablation_bankcap.pkl", 'wb') as f:
    pickle.dump(cap_results, f)
print("\n✓ Saved hitl_ablation_bankcap.pkl")

print(f"\n{'='*55}")
print(f"{'Max bank':>10} {'Final AUROC':>12} {'AUCC':>8} {'FP':>5} {'FN':>5}")
print('='*55)
for cap, r in cap_results.items():
    print(f"{cap:>10} {r['final_auroc']:>12.4f} {r['aucc']:>8.4f} "
          f"{r['total_fp_corrected']:>5} {r['total_fn_corrected']:>5}")


  max_bank = 2500
    Baseline AUROC: 0.8861  (max_bank=2500)
    Round   1 | cap=2500 | FP (good                ) | AUROC: 1.0000 | bank: 1254
    Round   2 | cap=2500 | FP (good                ) | AUROC: 0.9972 | bank: 2038
    Round   3 | cap=2500 | FP (good                ) | AUROC: 0.9917 | bank: 2500
    Round   4 | cap=2500 | FP (good                ) | AUROC: 0.9750 | bank: 2500
    Round   5 | cap=2500 | FP (good                ) | AUROC: 0.9833 | bank: 2500
    Round   6 | cap=2500 | FN (defective           ) | AUROC: 0.9861 | bank: 2495
    Round   7 | cap=2500 | FP (good                ) | AUROC: 0.9861 | bank: 2500
    Round   8 | cap=2500 | FN (defective           ) | AUROC: 0.9917 | bank: 2495
    Round   9: no errors — stopping early

    cap=2500: AUCC=0.9203  Final=0.9917

  max_bank = 5000
    Baseline AUROC: 0.8861  (max_bank=5000)
    Round   1 | cap=5000 | FP (good                ) | AUROC: 1.0000 | bank: 1254
    Round   2 | cap=5000 | FP (good                ) 